In [2]:
import itertools
import json
import os
import subprocess
import re
from datetime import datetime
import pandas as pd
from model import *
from model_segment import *
from dataloader import *
import numpy as np
import torch
import pickle
import random
from utils import *

/Users/shenjiajun/miniconda3/envs/ml2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# PARAMETERS

NUM_LAYERS = 6
HIDDEN_DIM = 512
GAMMA = 2.0
EPOCHS = 30
TRAIN_SAMPLES = None # None for the whole test set, and specify a smaller number to run test
VAL_NUM = 2000
CONTEXT = True
COMBINED_LOSS = True

In [ ]:
def train(split = 'test', num_classes = 11, train_type = 'token', random_seed=42, device ='cuda', cache_dir='.cache', data_cache='data_cache.pkl', train_split=0.8, hidden_dim=512, num_layers=6, num_heads = 8, max_len=512, dropout = 0.2, weight_decay =0.01, learning_rate=2e-4, gamma=2.0, batch_size=16, epochs=30, model_save_dir='models', end_idx = 10, context = True, combined_loss = True):
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)

    # Loading dataset and extracting features
    os.makedirs(f"{cache_dir}/{train_type}", exist_ok=True)
    feature_path = f"{cache_dir}/{train_type}/features.pkl"
    cache_path = f"{cache_dir}/{train_type}/cache.pkl"
 
    print("Loading dataset...")
    dataset = DocLayNetDataset(split=split, rewrite_storage=False, end_idx=end_idx)
    assert train_type in ['token', 'gold_seg', 'noisy_seg'], "Invalid type specified."
    feature_extractor = FeatureExtractor(dataset, context=context) if train_type == 'token' else FeatureExtractorSegment(dataset, context=context)
    if os.path.exists(feature_path):
        print(f"Found feature cache: {feature_path}, Loading cached features...")
        with open(feature_path, 'rb') as f:
            cache_data = pickle.load(f)
            pages_features = cache_data['pages_features']
            pages_targets = cache_data['pages_targets']
            feature_dim = cache_data['feature_dim']
        print(f"Loaded.")
    else:
        if os.path.exists(cache_path):
            print("Loading data cache...")
            dataset.load_cache(input_path=cache_path)

        feature_extractor = FeatureExtractor(dataset, context=context) if train_type == 'token' else FeatureExtractorSegment(dataset, context=context)
        if not os.path.exists(cache_path):
            dataset.save_cache(output_path=cache_path)
        pages_features, pages_targets = feature_extractor.get_page_features()
        feature_dim = len(pages_features[0][0])
        cache_data = {
        'pages_features': pages_features,
        'pages_targets': pages_targets,
        'feature_dim': feature_dim
    }
    print(f"Feature dimension: {feature_dim}")
    with open(feature_path, 'wb') as f:
        pickle.dump(cache_data, f)

    # Truncate to max_len
    pages_features = [p[:max_len] for p in pages_features]

    # If training type is noisy_seg, first replace the features with predictions from a pretrained token type model
    if train_type == 'noisy_seg':
        print("Replacing with predictions")
        token_type_model = TransformerTagger(
            input_dim=feature_dim - 3 if context else feature_dim - 1,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            output_dim=len(TokenType),
            dropout=dropout,
            max_seq_len=max_len,
        ).to(device)
        # Load pretrained token type model, the naming is consistent with the one used in training
        try:
            token_type_model.load_state_dict(torch.load(f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}/model_token_epoch{epochs}.pth", map_location=device))
        except FileNotFoundError:
            print("Pretrained token type model not found, please train the token type model first or ensure the arguments match an existing model.")
            return
        # token_type_model.load_state_dict(torch.load("models/best_larger_train_deep.pth", map_location=device)) # for testing
        token_type_model.eval()
        with open(f"{cache_dir}/token/features.pkl", 'rb') as f:
            features = pickle.load(f)
        features, _feature_dim = features.get('pages_features'), features.get('feature_dim')
        indices, length = feature_extractor.identify_adding_positions()
        print(f"Identified {len(indices)} positions to add replace")
        assert length == _feature_dim + 3 if context else _feature_dim + 1, "Length mismatch when adding noisy segment features."
        pages_labels = []
        ls = []
        for feature in tqdm(features):
            feature = torch.tensor(feature[:max_len], dtype=torch.float32).to(device)
            prediction = token_type_model(feature)
            ls.append(len(feature))
            labels = torch.argmax(prediction, dim=-1)
            pages_labels.extend(labels.cpu().numpy().tolist())
        # token type
        predictions = pages_labels
        # previous token type
        prev_predictions = [0.0] + predictions[:-1]
        # next token type
        next_predictions = predictions[1:] + [0.0]
        # save lengths to recover features later
        each_length = [min(max_len, len(p)) for p in pages_features]
        # print(each_length)
        # print(ls)
        assert sum(each_length) == len(predictions), f"Length mismatch in predictions: {sum(each_length)} != {len(predictions)}"
        pages_features_stacked = np.vstack(pages_features)
        # replace the identified positions with predictions
        if context:
            pages_features_stacked = replace_columns(pages_features_stacked, [predictions, prev_predictions, next_predictions], ks=indices)
        else:
            pages_features_stacked = replace_columns(pages_features_stacked, [predictions], ks=indices)

        pages_features = []
        _start = 0
        # recover page-wise features
        for k in each_length:
            pages_features.append(pages_features_stacked[_start:_start+k, :])
            _start += k

    features_and_targets = list(zip(pages_features, pages_targets))
    random.shuffle(features_and_targets)
    total = len(features_and_targets)
    training_data = features_and_targets[:int(train_split * total)]
    validation_data = features_and_targets[int(train_split * total):]
    print(f"\nTraining samples: {len(training_data)}, Validation samples: {len(validation_data)}")
    
    class_weights = compute_class_weights(pages_targets, num_classes, device)
    model = TransformerTagger(
        input_dim=feature_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        # num_layers=4, for testing ---
        num_heads=num_heads,
        output_dim=num_classes,
        dropout=dropout,
        max_seq_len=max_len, 
        # max_seq_len=2000, for testing ---
    ).to(device)
    # model.load_state_dict(torch.load("models/best_model_segment_.pth", map_location=device)) for testing ---
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    
    criterion = CombinedLoss(
        num_classes=num_classes,
        alpha=class_weights,
        gamma=gamma,
        label_smoothing=0.1,
        ignore_index=-1
    ) if COMBINED_LOSS else nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
    print(criterion)
    BATCH_SIZE = batch_size
    EPOCHS = epochs
    
    best_accuracy = 0.0
    losses_history = []
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_losses = []
        
        pbar = tqdm(range(0, len(training_data), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}")
        for i in pbar:
            batch_data = training_data[i:i+BATCH_SIZE]
            
            x, y, masks, lengths = collate_batch(batch_data, feature_dim, max_seq_len=max_len)
            x, y, masks = x.to(device), y.to(device), masks.to(device)
            
            optimizer.zero_grad()
            pred = model(x, mask=masks)
            
            pred_flat = pred.view(-1, num_classes)
            y_flat = y.view(-1)
            
            loss = criterion(pred_flat, y_flat)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            epoch_losses.append(loss.item())
            losses_history.append(loss.item())
            
            if len(epoch_losses) > 0:
                pbar.set_postfix({'loss': f'{np.mean(epoch_losses[-100:]):.4f}'})
        
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            if train_type != 'noisy_seg':
                # mkdir "H{hidden_dim}_L{num_layers}_G{gamma}"
                os.makedirs(f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}", exist_ok=True)
                torch.save(model.state_dict(), f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}/model_{train_type}_epoch{epoch+1}.pth")
            accuracy, class_acc, _, _ = evaluate_model(model, validation_data, feature_dim, device, num_classes, max_len=max_len)
            print(f"Epoch {epoch+1} Validation Accuracy: {accuracy:.4f}")
    # For noisy_seg, the epoch of token type model should match the one used here, so only save at the end
    if train_type == 'noisy_seg': # only save at the end for noisy_seg
        os.makedirs(f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}", exist_ok=True)
        torch.save(model.state_dict(), f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}/model_{train_type}_epoch{epochs}.pth")
        accuracy, class_acc, _, _ = evaluate_model(model, validation_data, feature_dim, device, num_classes, max_len=max_len)
        print(f"Epoch {epochs} Final Validation Accuracy: {accuracy:.4f}")


In [4]:
# First, train the token_type model

train(
    split="test",
    num_classes=11,
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    train_type="token",
    cache_dir=".cache",
    train_split=0.8,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    max_len=512,
    learning_rate=2e-4,
    gamma=GAMMA,
    batch_size=16,
    epochs=EPOCHS,
    model_save_dir="models",
    end_idx=TRAIN_SAMPLES,
    context=CONTEXT,
    combined_loss=COMBINED_LOSS
)

Loading dataset...
Found feature cache: .cache/token/features.pkl, Loading cached features...
Loaded.
Feature dimension: 31

Training samples: 8, Validation samples: 2
Label distribution:
  Class 0: 0 samples (0.00%), weight: 93454552.0000
  Class 1: 6 samples (0.58%), weight: 15.5758
  Class 2: 4 samples (0.39%), weight: 23.3636
  Class 3: 607 samples (59.05%), weight: 0.1540
  Class 4: 0 samples (0.00%), weight: 93454552.0000
  Class 5: 0 samples (0.00%), weight: 93454552.0000
  Class 6: 365 samples (35.51%), weight: 0.2560
  Class 7: 6 samples (0.58%), weight: 15.5758
  Class 8: 23 samples (2.24%), weight: 4.0632
  Class 9: 3 samples (0.29%), weight: 31.1515
  Class 10: 14 samples (1.36%), weight: 6.6753
CombinedLoss()


Epoch 10/10: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s, loss=0.9162]

Epoch 10 Validation Accuracy: 0.0000


In [ ]:
# Then, train the segmenting model

train(
    split="test",
    num_classes=2,
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    train_type="noisy_seg", # or "seg_noisy"
    cache_dir=".cache",
    train_split=0.8,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    max_len=512,
    learning_rate=2e-4,
    gamma=GAMMA,
    batch_size=16,
    epochs=EPOCHS,
    model_save_dir="models",
    end_idx=TRAIN_SAMPLES,
    context=CONTEXT,
    combined_loss=COMBINED_LOSS
)

Loading dataset...
Found feature cache: .cache/noisy_seg/features.pkl, Loading cached features...
Loaded.
Feature dimension: 32
Replacing with predictions


Extracting features:   0%|          | 0/4999 [00:00<?, ?it/s]


Identified 1 positions to add replace


100%|██████████| 10/10 [00:00<00:00, 54.82it/s]



Training samples: 8, Validation samples: 2
Label distribution:
  Class 0: 893 samples (86.87%), weight: 0.5756
  Class 1: 135 samples (13.13%), weight: 3.8074


Epoch 10/10: 100%|██████████| 1/1 [00:00<00:00,  1.24it/s, loss=0.2314]

Epoch 10 Validation Accuracy: 0.1631
Epoch 10 Final Validation Accuracy: 0.1631


In [ ]:
# test_results
def aggregating_tokens(tokens, segs):
    assert len(tokens) == len(segs), "Length mismatch between tokens and segments."

    curr_tokens = []
    starting_index = 0
    for i, token, seg in zip(range(len(tokens)), tokens, segs):
        if seg == 0:
            curr_tokens.append(token)
        else:
            curr_tokens.append(token)
            majority_token = max(set(curr_tokens), key=curr_tokens.count)
            tokens[starting_index:i+1] = [majority_token] * (i - starting_index + 1)
            starting_index = i + 1
            curr_tokens = []

    if curr_tokens:
        majority_token = max(set(curr_tokens), key=curr_tokens.count)
        tokens[starting_index:len(tokens)] = [majority_token] * (len(tokens) - starting_index)
    return tokens, segs

def evaluate_model(model, segment_model, validation_data, device, num_classes, max_len=512, extractor=None):
    model.eval()
    segment_model.eval()
    all_preds_token = []
    all_preds_seg = []
    all_targets_token = []
    all_targets_seg = []
    import time
    start_time = time.time()
    with torch.no_grad():
        for page_features, targets_token, targets_seg in validation_data:
            x = torch.tensor(page_features, dtype=torch.float32).to(device)
            y_token = torch.tensor(targets_token, dtype=torch.long).to(device)
            # print(x.shape)
            if x.shape[0] > max_len:
                x = x[:max_len, :] 

            x = x.unsqueeze(0)
            
            if y_token.shape[0] > max_len:
                y_token = y_token[:max_len]


            token_pred = model(x)
            _, predicted = torch.max(token_pred.squeeze(0), dim=1)

            all_preds_token.extend(predicted.cpu().tolist())
            all_targets_token.extend(y_token.tolist())
    end_time = time.time()
    print(f"Token type evaluation took {end_time - start_time} seconds, average per page: {(end_time - start_time)/len(validation_data)} seconds")
    correct = sum(p == t for p, t in zip(all_preds_token, all_targets_token))
    accuracy = correct / len(all_targets_token)

    class_correct = [0] * num_classes
    class_total = [0] * num_classes
    
    for p, t in zip(all_preds_token, all_targets_token):
        class_total[t] += 1
        if p == t:
            class_correct[t] += 1
    
    class_acc = [class_correct[i] / (class_total[i] + 1e-6) for i in range(num_classes)]
    indices, length = extractor.identify_adding_positions()
    print(f"Identified {len(indices)} positions to add replace")
    all_preds_token_prev = [0.0] + all_preds_token[:-1]
    all_preds_token_next = all_preds_token[1:] + [0.0]

    each_len = [min(len(p), max_len) for p, _, _ in validation_data]
    assert sum(each_len) == len(all_preds_token), f"Length mismatch in predictions: {sum(each_len)} != {len(all_preds_token)}"
    
    pages_features_truncated = [p[:max_len] for p, _, _ in validation_data]
    pages_features_stacked = np.vstack(pages_features_truncated)
    if extractor.context:
        pages_features_stacked = insert_columns(pages_features_stacked, [all_preds_token, all_preds_token_prev, all_preds_token_next], ks=indices)
    else:
        pages_features_stacked = insert_columns(pages_features_stacked, [all_preds_token], ks=indices)
    # print(pages_features_stacked.shape)
    pages_features = []
    _start = 0
    for k in each_len:
        pages_features.append(pages_features_stacked[_start:_start+k, :])
        _start += k
    
    for i, features in enumerate(pages_features):
        targets_seg = validation_data[i][2][:max_len]
        x_seg = torch.tensor(features, dtype=torch.float32).to(device)
        y_seg = torch.tensor(targets_seg, dtype=torch.long).to(device)
        if x_seg.shape[0] > max_len:
            x_seg = x_seg[:max_len, :]
        if y_seg.shape[0] > max_len:
            y_seg = y_seg[:max_len]
        x_seg = x_seg.unsqueeze(0)
        seg_pred = segment_model(x_seg)
        _, predicted_seg = torch.max(seg_pred.squeeze(0), dim=1)
        all_preds_seg.extend(predicted_seg.cpu().tolist())
        all_targets_seg.extend(y_seg.tolist())
    
    seg_correct = sum(p == t for p, t in zip(all_preds_seg, all_targets_seg))
    seg_accuracy = seg_correct / len(all_targets_seg)

    seg_class_correct = [0] * 2
    seg_class_total = [0] * 2
    for p, t in zip(all_preds_seg, all_targets_seg):
        seg_class_total[t] += 1
        if p == t:
            seg_class_correct[t] += 1
    seg_class_accuracy = [seg_class_correct[i] / (seg_class_total[i] + 1e-6) for i in range(2)]

    # print(all_preds_token)

    all_preds_token, all_preds_seg = aggregating_tokens(all_preds_token, all_preds_seg)
    correct_token = sum(p == t for p, t in zip(all_preds_token, all_targets_token))
    accuracy_token = correct_token / len(all_targets_token)

    class_correct_token = [0] * num_classes
    class_total_token = [0] * num_classes
    
    for p, t in zip(all_preds_token, all_targets_token):
        class_total_token[t] += 1
        if p == t:
            class_correct_token[t] += 1
    
    class_acc_token = [class_correct_token[i] / (class_total_token[i] + 1e-6) for i in range(num_classes)]
    # print(all_preds_token)
    # print(all_preds_seg)

    return accuracy, class_acc, all_preds_token, all_targets_token, seg_accuracy, seg_class_accuracy, accuracy_token, class_acc_token



def evaluate(split = 'validation', num_classes = 11, val_type = 'token', random_seed=42, device ='cuda', validation_cache = '.cache/val.pkl', train_split=0.8, hidden_dim=512, num_layers=6, num_heads = 8, max_len=512, dropout = 0.2, weight_decay =0.01, learning_rate=2e-4, gamma=2.0, batch_size=16, epochs=30, model_save_dir='models', end_idx = 10, results_dir = '.results', context = True):
    extractor = FeatureExtractor(DocLayNetDataset(split=split, rewrite_storage=False, end_idx=end_idx), context=context)
    extractor_segment = FeatureExtractorSegment(DocLayNetDataset(split=split, rewrite_storage=False, end_idx=end_idx), context=context)

    if not os.path.exists(validation_cache):
        pages_features, pages_targets_token = extractor.get_page_features()
        pages_features_seg, pages_targets_seg = extractor_segment.get_page_features()
        saving = {
            'pages_features': pages_features,
            'pages_targets_token': pages_targets_token,
            'pages_features_seg': pages_features_seg,
            'pages_targets_seg': pages_targets_seg
        }
        pickle.dump(saving, open(validation_cache, 'wb'))
    else:
        saving = pickle.load(open(validation_cache, 'rb'))
        pages_features = saving['pages_features']
        pages_targets_token = saving['pages_targets_token']
        pages_features_seg = saving['pages_features_seg']
        pages_targets_seg = saving['pages_targets_seg']
        
    validation_data = list(zip(pages_features, pages_targets_token, pages_targets_seg))
    model = TransformerTagger(
        input_dim=len(pages_features[0][0]),
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_heads=num_heads,
        output_dim=num_classes,
        dropout=dropout,
        max_seq_len=max_len, 
    ).to(device)
    # model.load_state_dict(torch.load(f"models_backup/best_larger_train_deep.pth", map_location=device))
    model.load_state_dict(torch.load(f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}/model_token_epoch{epochs}.pth", map_location=device))

    model_segment = TransformerTagger(
        input_dim=len(pages_features_seg[0][0]),
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        # num_layers=4,
        num_heads=num_heads,
        output_dim=2,
        dropout=dropout,
        max_seq_len=max_len, 
    ).to(device)
    # model_segment.load_state_dict(torch.load(f"best_segment_model.pth", map_location=device))
    model_segment.load_state_dict(torch.load(f"{model_save_dir}/H{hidden_dim}_L{num_layers}_G{str(gamma).replace('.', '')}/model_{val_type}_epoch{epochs}.pth", map_location=device))

    accuracy, class_acc, all_preds_token, all_targets_token, seg_accuracy, seg_class_accuracy, accuracy_token, class_acc_token = evaluate_model(model, model_segment, validation_data, device, num_classes, max_len=max_len, extractor=extractor_segment)
    
    if results_dir:
        os.makedirs(results_dir, exist_ok=True)
        with open(f"{results_dir}/results_{val_type}_L{num_layers}_H{hidden_dim}_G{str(gamma).replace('.', '')}_epoch{epochs}.txt", "w") as f:
            f.write(f"Validation Accuracy: {accuracy:.4f}\n")
            f.write(f"Validation Class Accuracy: {class_acc}\n")
            f.write(f"Segmentation Accuracy: {seg_accuracy:.4f}\n")
            f.write(f"Segmentation Class Correct: {seg_class_accuracy}\n")
            f.write(f"Token Accuracy: {accuracy_token:.4f}\n")
            f.write(f"Token Class Accuracy: {class_acc_token}\n")


    return accuracy, class_acc, all_preds_token, all_targets_token, seg_accuracy, seg_class_accuracy, accuracy_token, class_acc_token

In [ ]:
accuracy, class_acc, all_preds_token, all_targets_token, seg_accuracy, seg_class_accuracy, accuracy_token, class_acc_token = evaluate(
    split="validation",
    num_classes=11,
    random_seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
    val_type="noisy_seg", # or "gold_seg"
    validation_cache = '.cache/val.pkl',
    train_split=0.8,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    max_len=512,
    learning_rate=2e-4,
    gamma=GAMMA,
    batch_size=16, # not used, just to make it the same as train
    epochs=EPOCHS,
    model_save_dir="models",
    end_idx=VAL_NUM,
    results_dir ='.results',
    context=CONTEXT
)
print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Class Accuracy: {class_acc}")
print(f"Segmentation Accuracy: {seg_accuracy:.4f}")
print(f"Segmentation Class Correct: {seg_class_accuracy}")
print(f"Token Accuracy: {accuracy_token:.4f}")
print(f"Token Class Accuracy: {class_acc_token}")

Token type evaluation took 30.899495363235474 seconds, average per page: 0.015558658289645253 seconds


Extracting features:   0%|          | 0/6489 [00:00<?, ?it/s]

Identified 3 positions to add replace


Validation Accuracy: 0.9039
Validation Class Accuracy: [0.9799667463814442, 0.8741007162802132, 0.8487356321188708, 0.9642131222937721, 0.8336189169236431, 0.8749999915865385, 0.8557790341926883, 0.9011570244954853, 0.8241502404432484, 0.8378881984975503, 0.9815331007032986]
Segmentation Accuracy: 0.9099
Segmentation Class Correct: [0.9116694192284451, 0.895465944791391]
Token Accuracy: 0.9031
Token Class Accuracy: [0.9809805750038127, 0.8633093494125563, 0.8450574711996124, 0.9652565110034791, 0.8262508566096212, 0.8653846070636095, 0.8564072460545576, 0.881652892270528, 0.8016451962126889, 0.8267080742774199, 0.9731707313682333]
